# 00. Подготовка данных

Скачивает и готовит все датасеты на Google Drive. Запускается **один раз**, дальше остальные ноутбуки берут данные с Drive.

| Датасет | Для чего | Что хранится на Drive |
| --- | --- | --- |
| Golos (test) | речь с расшифровками для модуля 1 | ~1 ГБ |
| MUSAN (noise, music) | фон для смесей | до 2 ГБ |
| Dusha (Hugging Face) | эмоции, модуль 2 | 2–4 ГБ |
| Speech Commands v0.02 | короткие слова, модуль 3 | ~0,3 ГБ (10 слов) |

Перед запуском: **Среда выполнения → Сменить среду → T4 GPU** (для подготовки данных GPU не нужен, но так не придётся перезапускаться).

In [18]:
# Подключаем Google Drive, скачиваем код и ставим зависимости (~2 минуты)
from google.colab import drive
drive.mount('/content/drive')

REPO = 'https://github.com/tanyalebedeva/call-audio-analysis.git'
!test -d /content/call-audio-analysis || git clone -q $REPO /content/call-audio-analysis
%cd /content/call-audio-analysis
!git pull -q
!pip install -q -r requirements.txt

DATA = '/content/drive/MyDrive/coursework/data'
RUNS = '/content/drive/MyDrive/coursework/runs'
!mkdir -p $DATA $RUNS
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo 'GPU не подключён: Среда выполнения → Сменить среду → T4 GPU'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/call-audio-analysis
name, memory.total [MiB]
Tesla T4, 15360 MiB


## Проверка кода на синтетике
За ~1 минуту проверяет, что обучение, чекпойнты и метрики работают в этой среде.

In [19]:
!python -m pytest -q

...............F.                                                        [100%]
=================================== FAILURES ===================================
___________________ test_emotion_trains_and_dumps_embeddings ___________________

tmp_path = PosixPath('/tmp/pytest-of-root/pytest-0/test_emotion_trains_and_dumps_0')

    def test_emotion_trains_and_dumps_embeddings(tmp_path):
        cfg = _cfg("configs/emotion/mel_cnn.yaml", tmp_path, "emo", **{
            "data.synthetic_size": 64, "data.max_sec": 1.0, "batch_size": 8, "loss.name": "focal_weighted"})
>       train(cfg)

tests/test_training_smoke.py:35: 
_ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ 
callaudio/train.py:91: in train
    loss, logs = task.loss(model, batch, epoch, step / len(train_loader))
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
callaudio/tasks.py:168: in loss
    loss = self.criterion(logits.float(), batch["label"])
           ^^^^^^^^^^^^^^

## 1. Golos - речь с расшифровками

In [20]:
!python -m callaudio.data.prepare golos --out $DATA

[download] https://sc.link/Kqr
[extract] golos_test.tar
Traceback (most recent call last):
  File "<frozen runpy>", line 203, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/call-audio-analysis/callaudio/data/prepare.py", line 329, in <module>
    main()
    ~~~~^^
  File "/content/call-audio-analysis/callaudio/data/prepare.py", line 315, in main
    prepare_golos(a.out, a.work)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^
  File "/content/call-audio-analysis/callaudio/data/prepare.py", line 118, in prepare_golos
    with tarfile.open(tar_path) as t:
         ~~~~~~~~~~~~^^^^^^^^^^
  File "/usr/lib/python3.13/tarfile.py", line 1956, in open
    raise ReadError(f"file could not be opened successfully:\n{error_msgs_summary}")
tarfile.ReadError: file could not be opened successfully:
- method gz: ReadError('empty file')
- method bz2: ReadError('not a bzip2 file')
- method xz: ReadError('not an lzma file')
- method tar: ReadError('empty file')


## 2. MUSAN - шум и музыка
Архив ~11 ГБ скачивается во временную папку Colab, на Drive кладётся только нужное.

In [21]:
!python -m callaudio.data.prepare musan --out $DATA --music-files 300

  0.6 / 10.3 ГБ
  0.8 / 10.3 ГБ
  1.0 / 10.3 ГБ
  1.2 / 10.3 ГБ
  1.4 / 10.3 ГБ
  1.6 / 10.3 ГБ
  1.8 / 10.3 ГБ
  2.0 / 10.3 ГБ
  2.1 / 10.3 ГБ
  2.3 / 10.3 ГБ
  2.5 / 10.3 ГБ
  2.7 / 10.3 ГБ
  2.9 / 10.3 ГБ
  3.1 / 10.3 ГБ
  3.3 / 10.3 ГБ
  3.5 / 10.3 ГБ
  3.7 / 10.3 ГБ
  3.9 / 10.3 ГБ
  4.1 / 10.3 ГБ
  4.3 / 10.3 ГБ
  4.5 / 10.3 ГБ
  4.7 / 10.3 ГБ
  4.9 / 10.3 ГБ
  5.1 / 10.3 ГБ
  5.3 / 10.3 ГБ
  5.5 / 10.3 ГБ
  5.7 / 10.3 ГБ
  5.9 / 10.3 ГБ
  6.1 / 10.3 ГБ
  6.2 / 10.3 ГБ
  6.4 / 10.3 ГБ
  6.6 / 10.3 ГБ
  6.8 / 10.3 ГБ
  7.0 / 10.3 ГБ
  7.2 / 10.3 ГБ
  7.4 / 10.3 ГБ
  7.6 / 10.3 ГБ
  7.8 / 10.3 ГБ
  8.0 / 10.3 ГБ
  8.2 / 10.3 ГБ
  8.4 / 10.3 ГБ
  8.6 / 10.3 ГБ
  8.8 / 10.3 ГБ
  9.0 / 10.3 ГБ
  9.2 / 10.3 ГБ
  9.4 / 10.3 ГБ
  9.6 / 10.3 ГБ
  9.8 / 10.3 ГБ
  10.0 / 10.3 ГБ
  10.2 / 10.3 ГБ
[extract] musan: noise и music (это несколько минут)
/content/call-audio-analysis/callaudio/data/prepare.py:151: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and r

## 3. Dusha - эмоции
6000 записей на класс в train (если в классе меньше - берутся все) и 1000 в test.

In [ ]:
!python -m callaudio.data.prepare dusha --out $DATA --per-class 6000 --test-per-class 1000

README.md: 100% 473/473 [00:00<00:00, 1.59MB/s]
Resolving data files: 100% 48/48 [00:00<00:00, 22153.01it/s]
  train: {'angry': 427, 'neutral': 1026, 'positive': 327, 'sad': 220}
  train: {'angry': 857, 'neutral': 2064, 'positive': 664, 'sad': 415}
  train: {'angry': 1286, 'neutral': 3106, 'positive': 1003, 'sad': 605}
  train: {'angry': 1716, 'neutral': 4190, 'positive': 1319, 'sad': 775}
  train: {'angry': 2130, 'neutral': 5250, 'positive': 1651, 'sad': 969}
  train: {'angry': 2706, 'neutral': 6000, 'positive': 2073, 'sad': 1221}
  train: {'angry': 3620, 'neutral': 6000, 'positive': 2766, 'sad': 1614}
  train: {'angry': 4545, 'neutral': 6000, 'positive': 3459, 'sad': 1996}
  train: {'angry': 5436, 'neutral': 6000, 'positive': 4168, 'sad': 2396}
  train: {'angry': 6000, 'neutral': 6000, 'positive': 5126, 'sad': 2874}
  train: {'angry': 6000, 'neutral': 6000, 'positive': 6000, 'sad': 4000}


## 4. Окна Dusha по 1 с - для VAE на русской речи (модуль 3 → признаки для модуля 2)

In [ ]:
!python -m callaudio.data.prepare dusha_windows --out $DATA

## 5. Speech Commands - короткие слова для VAE

In [ ]:
!python -m callaudio.data.prepare commands --out $DATA --words core

## 6. Анализ данных - графики для главы 2 записки

In [ ]:
import json, collections, numpy as np, matplotlib.pyplot as plt
from pathlib import Path
M = Path(DATA) / 'manifests'
def load(name): return [json.loads(l) for l in open(M / name, encoding='utf-8')]

fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
emo = load('emotion_train.jsonl')
cnt = collections.Counter(r['label'] for r in emo)
axes[0].bar(cnt.keys(), cnt.values()); axes[0].set_title('Dusha (train): записей по эмоциям')
axes[1].hist([r['duration'] for r in emo], bins=40); axes[1].set_title('Dusha: длительность, с')
speech = load('speech_train.jsonl')
axes[2].hist([r['duration'] for r in speech], bins=40); axes[2].set_title('Golos: длительность, с')
plt.tight_layout(); plt.savefig(f'{DATA}/data_overview.png', dpi=150); plt.show()
for name in sorted(p.name for p in M.glob('*.jsonl')):
    rows = load(name)
    print(f"{name:32s} {len(rows):7d} записей  {sum(r.get('duration', 0) for r in rows) / 3600:6.2f} ч")

In [ ]:
import torch
from callaudio.audio import load_audio, LogMel, EMOTION_MEL, vocoder_mel
row = next(r for r in emo if r['label'] == 'angry')
wav = load_audio(row['path'])[:16000 * 3]
fig, axes = plt.subplots(3, 1, figsize=(10, 7))
axes[0].plot(np.arange(len(wav)) / 16000, wav, lw=0.5); axes[0].set_title('Waveform')
spec = torch.stft(torch.from_numpy(wav), 512, 128, window=torch.hann_window(512), return_complex=True).abs()
axes[1].imshow(torch.log(spec + 1e-5), origin='lower', aspect='auto', cmap='magma'); axes[1].set_title('Спектрограмма STFT (лог-амплитуда)')
axes[2].imshow(LogMel(EMOTION_MEL)(torch.from_numpy(wav)[None])[0], origin='lower', aspect='auto', cmap='magma'); axes[2].set_title('Лог-мел-спектрограмма, 64 полосы')
plt.tight_layout(); plt.savefig(f'{DATA}/representations.png', dpi=150); plt.show()